# 02 · ACDC: the minimal circuit

**Question:** which of the EAP-IG candidate edges (Phase 1b) are really needed?

**Start.** The candidate circuit of Phase 1b (`candidate_mask.npy`, faithfulness ≈ 1 in both experiments).

**ACDC** (Conmy et al., 2023). Go through the receivers from the answer backwards (logits, m27, a27, m26, …) and
try to remove their incoming edges (weakest EAP-IG score first). An edge is removed for good if the circuit still explains the behaviour:

$$\text{loss} = \max_{\text{experiment}} \left|1 - \overline{F}\right| ,\qquad
\text{remove if}\quad \text{loss}_{\text{without}} - \text{loss}_{\text{with}} < \tau \ \text{ and }\ \text{loss}_{\text{without}} \le \text{cap}$$

(cap = starting loss + 0.05, at least 0.15).

$\overline{F}$ is the mean faithfulness over the ACDC pairs of one experiment; using $|1-F|$ penalises both losing
and overshooting the behaviour (like the KL divergence of the original ACDC).

**Made fast enough for a T4:**
* *batched evaluation*: all ACDC pairs of both experiments go through one forward pass per test;
* *hierarchical removal*: first try to remove **all** incoming edges of a receiver at once; if rejected, split
  them in two halves (ordered by layer) and recurse down to single edges. Unimportant groups go with one test;
* receivers that no longer reach the answer lose their incoming edges without a test.

We run several $\tau$ (each starting from the circuit of the previous one), then check every circuit on pairs
**not used by ACDC** (the held-out split and unused discovery pairs) against EAP-IG circuits of the same size
and random circuits of the same size.

**Runtime:** T4 GPU, Qwen2.5-7B-Instruct, a few hours. Progress is saved on Google Drive after every receiver:
if Colab stops, run all cells again and ACDC continues where it stopped.

## Setup

In [1]:
# Get (or update) the code from GitHub
import os, sys
REPO = "https://github.com/vasilis-pantzos/sarcasm-proposal.git"
ROOT = "/content/sarcasm-proposal"
if not os.path.exists(ROOT):
    !git clone -q {REPO} {ROOT}
else:
    !git -C {ROOT} pull -q
os.chdir(ROOT)
sys.path.insert(0, ROOT)
%pip install -q -U accelerate bitsandbytes

In [2]:
import os, json, time, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display
from src.model import load_model, model_info, DEV_MODEL, MAIN_MODEL
from src.prompts import load_dataset, make_pairs
from src.patching import score_pairs, keep_correct, mean_and_se
from src.eap import build_graph, top_k_mask, random_mask, edge_table, prune_dangling
from src.acdc import BatchEvaluator, acdc, faithfulness_many
from src.utils import mount_drive, results_dir, cached, set_seed

# ---------------- settings ----------------
MODEL = MAIN_MODEL          # DEV_MODEL (3B) for quick tests
N_ACDC = 8                  # pairs per experiment used by ACDC (all in one batch)
TAUS = [0.01]               # ACDC thresholds; more values (e.g. [0.01, 0.02]) run in order, each from the previous circuit
CAP_MARGIN = 0.05           # never accept a circuit whose loss is more than this above the starting loss (and above 0.10)
N_VAL_DISC = 24             # unused discovery pairs per experiment for validation (plus all held-out pairs)
TARGET_F = 0.9
# ------------------------------------------
TAG = MODEL.split("/")[-1]
SUB = f"02_acdc/{TAG}"
SUB_1B = f"01b_eap_ig/{TAG}"
mount_drive()
print("results go to:", results_dir(SUB))
set_seed(0)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
results go to: /content/drive/MyDrive/sarcasm/02_acdc/Qwen2.5-7B-Instruct


In [3]:
t0 = time.time()
model, tok = load_model(MODEL)
INFO = model_info(model)
G = build_graph(model)
print(MODEL, INFO, f"loaded in {(time.time() - t0) / 60:.1f} min")

# Phase 1b results
CAND = np.load(os.path.join(results_dir(SUB_1B), "candidate_mask.npy"))
SUM_1B = json.load(open(os.path.join(results_dir(SUB_1B), "summary.json")))
COMBINED = np.load(os.path.join(results_dir(SUB_1B), "mean_scores.npz"))["combined"]
print(f"candidate circuit from Phase 1b: {int(CAND.sum())} of {G.n_edges()} edges (top-{SUM_1B['k_star']})")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen/Qwen2.5-7B-Instruct {'n_layers': 28, 'n_heads': 28, 'n_kv_heads': 4, 'd_model': 3584, 'head_dim': 128} loaded in 1.2 min
candidate circuit from Phase 1b: 9600 of 23577 edges (top-9600)


## Pairs: for ACDC and for validation

In [4]:
df = load_dataset()
ACDC_PAIRS, VAL = [], {"heldout": [], "discovery (unused)": []}
for e in ["situation", "phrase"]:
    disc = keep_correct(score_pairs(model, tok, make_pairs(df, tok, e)))           # same pairs and order as 01b
    held = keep_correct(score_pairs(model, tok, make_pairs(df, tok, e, split="heldout")))
    ACDC_PAIRS += disc[:N_ACDC]
    VAL["discovery (unused)"] += disc[16:16 + N_VAL_DISC]                         # 01b measured faithfulness on the first 16
    VAL["heldout"] += held
    print(f"{e:9s}: {len(disc)} discovery pairs (ACDC uses {N_ACDC}), {len(held)} held-out pairs")
print("validation:", {k: len(v) for k, v in VAL.items()})

situation: 114 discovery pairs (ACDC uses 8), 28 held-out pairs
phrase   : 150 discovery pairs (ACDC uses 8), 0 held-out pairs
validation: {'heldout': 28, 'discovery (unused)': 48}


## Sanity check

All edges → $F = 1$, no edges → $F = 0$ (exactly), and the candidate circuit gives the same $F$ per pair as in
Phase 1b (there each pair was evaluated alone; here all pairs go in one batch, so tiny float16 differences are normal).

In [5]:
EV = BatchEvaluator(model, tok, ACDC_PAIRS, G)
F_all, F_none, F_cand = EV.faithfulness(G.valid), EV.faithfulness(np.zeros_like(G.valid)), EV.faithfulness(CAND)
CHECKS = {"all edges = 1": np.abs(F_all - 1).max() < 1e-3, "no edges = 0": np.abs(F_none).max() < 1e-3}
print(f"all edges: max |F - 1| = {np.abs(F_all - 1).max():.5f} | no edges: max |F| = {np.abs(F_none).max():.5f}")

curve = pd.read_csv(os.path.join(results_dir(SUB_1B), "faithfulness_curve.csv"))
j = list(curve.k).index(SUM_1B["k_star"])
gaps = []
for e in ["situation", "phrase"]:
    f1b = torch.load(os.path.join(results_dir(SUB_1B), f"faithfulness_{e}_{SUM_1B['n_eval']}pairs_kmax{curve.k.iloc[-1]}.pt"),
                     weights_only=False)["top"][:N_ACDC, j]
    here = F_cand[EV.exp == e][:len(f1b)]
    gaps.append(abs(here.mean() - f1b.mean()))
    print(f"{e:9s}: F of the candidate, batched {here.mean():.3f} vs Phase 1b {f1b.mean():.3f} "
          f"(per pair the float16 batch noise is up to {np.abs(here - f1b).max():.3f})")
CHECKS["same F as Phase 1b"] = max(gaps) < 0.02      # ACDC uses the mean over pairs
t0 = time.time(); START_LOSS = EV.loss(CAND); dt = time.time() - t0
CAP = float(np.ceil((max(0.10, START_LOSS) + CAP_MARGIN) * 100) / 100)
print(f"one ACDC test = {dt:.2f} s; start loss = {START_LOSS:.3f}, mean F = {EV.mean_F(CAND)} -> cap = {CAP}")
for k, v in CHECKS.items():
    print("PASS" if v else "FAIL", k)
assert all(CHECKS.values()), "sanity check failed: send the output to Claude"

all edges: max |F - 1| = 0.00000 | no edges: max |F| = 0.00000
situation: F of the candidate, batched 0.931 vs Phase 1b 0.936 (per pair the float16 batch noise is up to 0.059)
phrase   : F of the candidate, batched 1.136 vs Phase 1b 1.142 (per pair the float16 batch noise is up to 0.067)
one ACDC test = 7.60 s; start loss = 0.136, mean F = {np.str_('phrase'): 1.136413335800171, np.str_('situation'): 0.9314300417900085} -> cap = 0.19
PASS all edges = 1
PASS no edges = 0
PASS same F as Phase 1b


## Run ACDC

One line per receiver: incoming edges before → after, number of tests, size and loss of the whole circuit.
The first $\tau$ takes longest; the next ones start from a smaller circuit.

In [ ]:
RES = {}
start = CAND
for tau in TAUS:
    print(f"\n===== tau = {tau} =====")
    t0 = time.time()
    state = os.path.join(results_dir(SUB), f"acdc_state_tau{tau}_n{N_ACDC}.pt")
    RES[tau] = acdc(EV, start, tau=tau, cap=CAP, state_path=state, scores=COMBINED)
    print(f"tau = {tau}: {int(RES[tau]['mask'].sum())} edges, loss {RES[tau]['loss']:.3f}, "
          f"{RES[tau]['n_tests']} tests, {(time.time() - t0) / 60:.1f} min this session")
    start = RES[tau]["mask"]
del EV; torch.cuda.empty_cache()


===== tau = 0.01 =====
  logits:   260 ->   21 edges in,   52 tests | circuit  9361 edges, loss 0.094 | 6.5 min
     m27:   196 ->    0 edges in,    1 tests | circuit  9165 edges, loss 0.079 | 6.6 min
     a27:   129 ->    1 edges in,   11 tests | circuit  9036 edges, loss 0.095 | 7.9 min
     m26:   160 ->    0 edges in,    0 tests | circuit  8876 edges, loss 0.095 | 7.9 min
     a26:   116 ->    3 edges in,   22 tests | circuit  8763 edges, loss 0.146 | 10.5 min
     m25:   141 ->    1 edges in,   15 tests | circuit  8623 edges, loss 0.163 | 12.3 min
     a25:    72 ->    0 edges in,    0 tests | circuit  8551 edges, loss 0.163 | 12.3 min
     m24:   141 ->    4 edges in,   13 tests | circuit  8414 edges, loss 0.130 | 13.7 min
     a24:    66 ->    0 edges in,    6 tests | circuit  8348 edges, loss 0.142 | 14.4 min
     m23:   163 ->    5 edges in,   24 tests | circuit  8187 edges, loss 0.130 | 17.0 min
     a23:   107 ->    4 edges in,   19 tests | circuit  8084 edges, loss 0.139 |

## Validation on pairs ACDC did not see

For every ACDC circuit: the EAP-IG top-$k$ edges with the same number of edges, and random circuits of the same
size taken **inside** the candidate (3 seeds). Also the full candidate.

In [ ]:
def val_masks():
    m = {"candidate": CAND}
    rng = np.random.default_rng(0)
    cand_idx = np.flatnonzero(CAND)
    for tau, r in RES.items():
        k = int(r["mask"].sum())
        m[f"acdc {tau}"] = r["mask"]
        m[f"eap-ig top {tau}"] = top_k_mask(COMBINED, G, k)
        for s in range(3):
            rm = np.zeros_like(CAND); rm.flat[rng.choice(cand_idx, size=k, replace=False)] = True
            m[f"random {tau} {s}"] = rm
    return m

def run_val():
    masks = val_masks()
    out = {}
    for split, ps in VAL.items():
        F = faithfulness_many(model, tok, ps, masks, G, batch=16)
        exp = np.array([p.experiment for p in ps])
        out[split] = {name: {e: F[name][exp == e] for e in ["situation", "phrase"]} for name in masks}
    return out

t0 = time.time()
VALF = cached(f"validation_taus{'_'.join(map(str, TAUS))}_n{N_ACDC}", run_val, sub=SUB)
print(f"done in {(time.time() - t0) / 60:.1f} min")

rows = []
for tau, r in RES.items():
    row = dict(tau=tau, edges=int(r["mask"].sum()))
    for split in VAL:
        for e in ["situation", "phrase"]:
            row[f"{split} {e} ACDC"] = VALF[split][f"acdc {tau}"][e].mean()
            row[f"{split} {e} EAP-IG"] = VALF[split][f"eap-ig top {tau}"][e].mean()
            row[f"{split} {e} random"] = np.mean([VALF[split][f"random {tau} {s}"][e].mean() for s in range(3)])
    rows.append(row)
TAB = pd.DataFrame(rows)
display(TAB.round(3).T)
for split in VAL:
    print(f"candidate ({int(CAND.sum())} edges), {split}:",
          {e: round(float(VALF[split]['candidate'][e].mean()), 3) for e in ["situation", "phrase"]})

### Size vs faithfulness (held-out pairs)

In [ ]:
cols = {"situation": "#2E7D32", "phrase": "#C62828"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, e in zip(axes, ["situation", "phrase"]):
    for kind, ls, mk in [("ACDC", "-", "o"), ("EAP-IG", "--", "s"), ("random", ":", "x")]:
        ax.plot(TAB.edges, TAB[f"heldout {e} {kind}"], ls, marker=mk, color=cols[e], label=kind)
    c = VALF["heldout"]["candidate"][e].mean()
    ax.scatter([CAND.sum()], [c], marker="*", s=150, color="black", label="candidate (1b)", zorder=3)
    for _, r in TAB.iterrows():
        ax.annotate(f"τ={r.tau}", (r.edges, r[f"heldout {e} ACDC"]), fontsize=7, xytext=(4, 4), textcoords="offset points")
    ax.axhline(1, c="gray", lw=0.6); ax.axhline(TARGET_F, c="gray", ls="--", lw=0.8); ax.axhline(0, c="gray", lw=0.6)
    ax.set_xscale("log"); ax.set_xlabel("edges in the circuit"); ax.set_title(f"{e} swap · held-out pairs")
axes[0].set_ylabel("faithfulness F"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "size_vs_faithfulness.png"), dpi=150); plt.show()

## The final circuit

The smallest ACDC circuit with mean $0.9 \le F \le 1.2$ on the held-out pairs of **both** experiments.

In [ ]:
# mean F in [0.9, 1.2] on the held-out pairs of both experiments (above 1.2 the circuit overshoots: not faithful either)
ok = [tau for tau in TAUS if all(TARGET_F <= TAB.set_index("tau").loc[tau, f"heldout {e} ACDC"] <= 1.2 for e in ["situation", "phrase"])]
TAU_STAR = min(ok, key=lambda t: int(RES[t]["mask"].sum())) if ok else TAUS[0]
CIRC = RES[TAU_STAR]["mask"]
print(f"final circuit: tau = {TAU_STAR}, {int(CIRC.sum())} edges ({CIRC.sum() / G.n_edges():.1%} of the graph, "
      f"{CIRC.sum() / CAND.sum():.1%} of the candidate)" + ("" if ok else "   (target not reached: smallest tau)"))

# bar chart: ACDC vs EAP-IG same size vs random, both validation sets
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, split in zip(axes, VAL):
    names = [("candidate", "candidate"), (f"acdc {TAU_STAR}", "ACDC"), (f"eap-ig top {TAU_STAR}", "EAP-IG same size")]
    x = np.arange(4)
    for i, e in enumerate(["situation", "phrase"]):
        vals = [VALF[split][n][e] for n, _ in names] + [np.concatenate([VALF[split][f"random {TAU_STAR} {s}"][e] for s in range(3)])]
        m = [v.mean() for v in vals]; se = [v.std(ddof=1) / np.sqrt(len(v)) for v in vals]
        ax.bar(x + (i - 0.5) * 0.38, m, 0.38, yerr=1.96 * np.array(se), color=cols[e], alpha=0.8, label=e, capsize=3)
    ax.set_xticks(x); ax.set_xticklabels([l for _, l in names] + ["random same size"], fontsize=8)
    ax.axhline(1, c="gray", lw=0.6); ax.axhline(TARGET_F, c="gray", ls="--", lw=0.8)
    ax.set_title(f"{split} pairs")
axes[0].set_ylabel("faithfulness F"); axes[0].legend()
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "final_circuit_faithfulness.png"), dpi=150); plt.show()

### Where are the edges of the final circuit?

In [ ]:
L = G.n_layers
def sender_layer(i):
    if i == 0: return 0
    if i <= L * G.n_heads: return 1 + (i - 1) // G.n_heads
    return 1 + (i - 1 - L * G.n_heads)
rows = np.array([sender_layer(i) for i in range(len(G.senders))])
counts = np.zeros((L + 1, len(G.receivers)))
np.add.at(counts, rows, CIRC.astype(float))
fig, ax = plt.subplots(figsize=(10, 4.5))
im = ax.imshow(counts, aspect="auto", origin="lower", cmap="viridis")
ax.set_xticks([0, L // 2, L, L + L // 2, 2 * L]); ax.set_xticklabels(["a0", f"a{L // 2}", "m0", f"m{L // 2}", "logits"])
ax.set_xlabel("receiver"); ax.set_ylabel("sender layer (0 = input embeddings)")
ax.set_title(f"{TAG}: edges of the final circuit (count) by sender layer and receiver")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "circuit_edges_by_layer.png"), dpi=150); plt.show()

# the components (nodes) of the circuit
out_deg, in_deg = CIRC.sum(1), CIRC.sum(0)
nodes = []
for i, name in enumerate(G.senders):
    if out_deg[i] == 0: continue
    if name.startswith("a") and "." in name:
        l = int(name[1:].split(".")[0]); recv_in = in_deg[G.recv_attn(l)]
    elif name.startswith("m"):
        l = int(name[1:]); recv_in = in_deg[G.recv_mlp(l)]
    else:
        l, recv_in = -1, 0
    nodes.append(dict(component=name, layer=l, edges_out=int(out_deg[i]), edges_into_its_block=int(recv_in),
                      eap_out=float(np.abs(COMBINED[i][CIRC[i]]).sum())))
NODES = pd.DataFrame(nodes).sort_values("eap_out", ascending=False).reset_index(drop=True)
heads = NODES[NODES.component.str.contains(r"\.h")]
print(f"components in the circuit: {len(NODES)}  ({len(heads)} heads, {NODES.component.str.match(r'^m').sum()} MLPs)")
print("heads per layer:", heads.groupby("layer").size().to_dict())
display(NODES.head(30).round(3))

### Drawing of the circuit (strongest edges)

In [ ]:
def draw(mask, scores, n_max=80, title=""):
    H = G.n_heads
    def pos(name):
        if name == "input": return (H / 2, -1)
        if name == "logits": return (H / 2, L)
        if name.startswith("a") and "." in name:
            l, h = name[1:].split(".h"); return (int(h), int(l))
        if name.startswith("a"): return (H / 2, int(name[1:]) - 0.3)
        return (H + 2, int(name[1:]) + 0.3)
    tab = edge_table({"s": scores}, G, mask).head(n_max)
    lim = np.abs(tab["combined"]).max()
    fig, ax = plt.subplots(figsize=(10, 9))
    for _, r in tab.iterrows():
        (x0, y0), (x1, y1) = pos(r.sender), pos(r.receiver)
        ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle="->", color="#1565C0" if r.combined > 0 else "#C62828",
                                    lw=0.5 + 3 * abs(r.combined) / lim, alpha=0.7))
    for n in set(tab.sender) | set(tab.receiver):
        x, y = pos(n); ax.scatter(x, y, s=25, c="black", zorder=3); ax.text(x + 0.2, y + 0.15, n, fontsize=7)
    ax.set_xlim(-1, H + 5); ax.set_ylim(-2, L + 1)
    ax.set_xlabel("head index (MLPs on the right)"); ax.set_ylabel("layer"); ax.set_title(title)
    plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "circuit_drawing.png"), dpi=150); plt.show()

draw(CIRC, COMBINED, n_max=80,
     title=f"{TAG}: final ACDC circuit, the 80 strongest of {int(CIRC.sum())} edges (blue: towards sarcasm, red: against)")

## Save (for Phase 3 and the comparison with the full model)

In [ ]:
d = results_dir(SUB)
np.save(os.path.join(d, "circuit_mask.npy"), CIRC)
for tau, r in RES.items():
    np.save(os.path.join(d, f"acdc_mask_tau{tau}.npy"), r["mask"])
edge_table({"eap_ig": COMBINED}, G, CIRC).to_csv(os.path.join(d, "circuit_edges.csv"), index=False)
NODES.to_csv(os.path.join(d, "circuit_nodes.csv"), index=False)
TAB.to_csv(os.path.join(d, "acdc_results.csv"), index=False)
json.dump(dict(model=MODEL, tau_star=TAU_STAR, cap=CAP, taus=TAUS, n_acdc_pairs=N_ACDC,
               n_edges=int(CIRC.sum()), n_candidate=int(CAND.sum()), n_graph=G.n_edges(),
               n_components=len(NODES), n_tests={str(t): r["n_tests"] for t, r in RES.items()},
               heldout_F={e: float(VALF["heldout"][f"acdc {TAU_STAR}"][e].mean()) for e in ["situation", "phrase"]}),
          open(os.path.join(d, "summary.json"), "w"), indent=1)
print("saved to", d)